# Week #3 - Data Preprocessing

## 1.0. Import libraries

In [1]:
import os
import pandas as pd
import numpy as np
# from sklearn.preprocessing import TargetEncoder

pd.options.mode.chained_assignment = None  # default='warn'

## 2.0. Read in the CSV files

In [2]:
# Define the directory where the CSV files are located
assets_directory = "./assets"

# Get an alphabetically sorted list of the 12 latest CRMLSSold CSV file paths in the assets directory
crmls_sold_csv_filepaths = sorted([
    os.path.join(assets_directory, csv_file)
    for csv_file in os.listdir(assets_directory)
    if csv_file.endswith(".csv")
])[-12:]

# Print each CRMLSSold CSV file names 
print("CRMLSSold CSV files found in the assets directory:")
for crmls_sold_csv_file in crmls_sold_csv_filepaths:
    print(f"• {os.path.basename(crmls_sold_csv_file)}")

# Read each CRMLSSold CSV file into a DataFrame and store them in a list
crmls_sold_dfs = [pd.read_csv(csv_file, low_memory=False) for csv_file in crmls_sold_csv_filepaths]

# Concatenate all of the DataFrames into a single DataFrame
crmls_sold_df = pd.concat(crmls_sold_dfs, ignore_index=True)
print("")
print(f"Loaded {len(crmls_sold_csv_filepaths)} CRMLSSold CSV files into a single DataFrame with shape: {crmls_sold_df.shape}")


CRMLSSold CSV files found in the assets directory:
• CRMLSSold202510.csv
• CRMLSSold202511.csv
• CRMLSSold202512.csv
• CRMLSSold202601.csv
• CRMLSSold202602.csv
• CRMLSSold202603.csv
• CRMLSSold202604.csv
• CRMLSSold202605.csv
• CRMLSSold202606.csv
• CRMLSSold202607.csv
• CRMLSSold202608.csv
• CRMLSSold202609.csv

Loaded 12 CRMLSSold CSV files into a single DataFrame with shape: (266809, 80)


In [3]:
# Display the DataFrame information
display(crmls_sold_df.info())
print("")

# Print the date range of the data in the DataFrame
print(f"Date range: {crmls_sold_df['CloseDate'].min()} to {crmls_sold_df['CloseDate'].max()}")
print("")

# Display the first few rows of the DataFrame
display(crmls_sold_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266809 entries, 0 to 266808
Data columns (total 80 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   BuyerAgentAOR                 251085 non-null  object 
 1   ListAgentAOR                  266725 non-null  object 
 2   Flooring                      156200 non-null  object 
 3   ViewYN                        239420 non-null  object 
 4   WaterfrontYN                  155 non-null     object 
 5   BasementYN                    4379 non-null    object 
 6   PoolPrivateYN                 238100 non-null  object 
 7   OriginalListPrice             266039 non-null  float64
 8   ListingKey                    266809 non-null  int64  
 9   ListAgentEmail                145619 non-null  object 
 10  CloseDate                     266809 non-null  object 
 11  ClosePrice                    266804 non-null  float64
 12  ListAgentFirstName            265678 non-nul

None


Date range: 2025-10-01 to 2026-09-30



,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName
0,NorthSanLuisObispo,NorthSanLuisObispo,NaN,NaN,NaN,NaN,NaN,NaN,479293596,newlin@pacificaCRE.com,...,NaN,NaN,NaN,NaN,93446,NaN,1807.0,NaN,NaN,NaN
1,TheInlandGateway,TheInlandGateway,NaN,True,NaN,NaN,NaN,12000.0,446914808,nleimers@hotmail.com,...,NaN,False,NaN,NaN,92325,0.0,14253.0,NaN,NaN,NaN
2,SanDiego,SanDiego,Wood,True,NaN,NaN,False,1695.0,445176588,mannybehar@yahoo.com,...,NaN,NaN,0.0,NaN,92126,0.0,NaN,NaN,NaN,NaN
3,SanDiego,SanDiego,NaN,False,NaN,NaN,False,950000.0,1145282039,chase@cromwellhomegroup.com,...,NaN,False,2.0,NaN,91901,NaN,NaN,NaN,NaN,NaN
4,SanDiego,SanDiego,NaN,False,NaN,NaN,False,970000.0,1145278097,rory@corneliusestates.net,...,NaN,False,2.0,NaN,92115,NaN,NaN,NaN,NaN,NaN


In [4]:
# Change the date columns (which include 'Date' in their names) to datetime format
date_columns = [col for col in crmls_sold_df.columns if 'Date' in col]
for col in date_columns:
    crmls_sold_df[col] = pd.to_datetime(crmls_sold_df[col], errors='coerce')


# Create a temporary 'CloseMonth' column in the DataFrame, which contains the year and month of the 'CloseDate'
crmls_sold_df['CloseMonth'] = pd.to_datetime(crmls_sold_df['CloseDate'], errors='coerce').dt.to_period('M')

## 2.0. Filter the DataFrame

In [5]:
# # Filter the DataFrame to only include rows where the 'PropertyType' is 'Residential' and 'PropertySubType' 
# # is 'SingleFamilyResidence'
# crmls_sold_df = crmls_sold_df[(crmls_sold_df['PropertyType'] == 'Residential') &
#                               (crmls_sold_df['PropertySubType'] == 'SingleFamilyResidence')]

# print(f"Shape of the filtered DataFrame: {crmls_sold_df.shape}")
# print("")
# display(crmls_sold_df.head())

In [6]:
crmls_sold_filtered_df = crmls_sold_df.copy()  # Create a copy of the original DataFrame to work with

### 2.1. Remove duplicates, impossible, and non-California values

In [7]:
# Update this cell to include log count and percentage of rows dropped for each filtering step
intial_row_count = len(crmls_sold_filtered_df)
print(f"Initial number of rows: {intial_row_count}")

# Drop duplicate rows based on 'ListingKey' and 'CloseDate' columns, keeping the first occurrence
crmls_sold_filtered_df = crmls_sold_filtered_df.drop_duplicates(subset=['ListingKey', 'CloseDate'], keep='first')
print(f"Number of rows after dropping duplicates: {len(crmls_sold_filtered_df)}")

# Drop rows where 'ClosePrice' is missing or less than or equal to zero
crmls_sold_filtered_df = crmls_sold_filtered_df.dropna(subset=['ClosePrice'])
crmls_sold_filtered_df = crmls_sold_filtered_df[crmls_sold_filtered_df['ClosePrice'] > 0]
print(f"Number of rows after dropping rows with missing or non-positive 'ClosePrice': {len(crmls_sold_filtered_df)}")

# Drop rows where 'CloseDate' is earlier than 'ListingContractDate'
crmls_sold_filtered_df = crmls_sold_filtered_df[crmls_sold_filtered_df['CloseDate'] >= crmls_sold_filtered_df['ListingContractDate']]
print(f"Number of rows after dropping rows where 'CloseDate' is earlier than 'ListingContractDate': {len(crmls_sold_filtered_df)}")

# Drop rows where there is zero or negative values in the 'LotSizeSquareFeet' column
crmls_sold_filtered_df = crmls_sold_filtered_df[crmls_sold_filtered_df['LotSizeSquareFeet'] > 0]
print(f"Number of rows after dropping rows with non-positive 'LotSizeSquareFeet': {len(crmls_sold_filtered_df)}")

# Anything else to filter out? Also, keep track of percentage of rows dropped after each filtering step
# AttachedGraage vs GarageSpaces, etc.
# MainLevelBedrooms vs BedroomsTotal, etc.


Initial number of rows: 266809
Number of rows after dropping duplicates: 266791
Number of rows after dropping rows with missing or non-positive 'ClosePrice': 266766
Number of rows after dropping rows where 'CloseDate' is earlier than 'ListingContractDate': 266700
Number of rows after dropping rows with non-positive 'LotSizeSquareFeet': 239410


In [8]:
# Display the value counts of the 'StateOrProvince' column in the DataFrame
display(crmls_sold_filtered_df["StateOrProvince"].value_counts())

CA    239385
FL         3
HI         3
NV         2
AZ         2
TX         2
NY         1
OS         1
KS         1
NM         1
ID         1
OH         1
OR         1
NJ         1
CO         1
VA         1
OK         1
Name: StateOrProvince, dtype: int64

In [9]:
# Keep samples from the DataFrame where the 'StateOrProvince' is 'CA'
crmls_sold_filtered_df = crmls_sold_filtered_df[crmls_sold_filtered_df["StateOrProvince"] == "CA"]

# Print new value counts of the 'StateOrProvince' column after filtering for 'CA'
print("Value counts of the 'StateOrProvince' column after filtering for 'CA':")
display(crmls_sold_filtered_df["StateOrProvince"].value_counts())

# Print the updated number of rows and percentage of total rows dropped after filtering for 'CA'
print(f"Number of rows after filtering for 'CA': {len(crmls_sold_filtered_df)}")

Value counts of the 'StateOrProvince' column after filtering for 'CA':


CA    239385
Name: StateOrProvince, dtype: int64

Number of rows after filtering for 'CA': 239385


### 2.2. Remove columns for feature configuartion

#### 2.2.1. Drop columns that may causee data leakage

These include columns containing the target variable or listing price, MLS status, and post-close information

In [10]:
# Store columns related to data leakage to drop
drop_columns = [
    "ClosePrice",
    "CloseDate",
    "ListPrice",
    "OriginalListPrice",
    "DaysOnMarket",
    "PurchaseContractDate",
    "ContractStatusChangeDate",
    "MlsStatus"
]

# Drop columns related to data leakage from the filtered DataFrame
crmls_sold_filtered_df = crmls_sold_df.drop(columns=drop_columns)

#### 2.2.2 Drop additional columns containing identifiers, agent, MLS, and other unnecessary information

In [11]:
# Store idenitifer/agent columns to drop
additional_drop_columns = [
    "ListingId",
    "ListingKey",
    "ListingKeyNumeric",
    
    "ListAgentAOR", 
    "ListAgentEmail",
    "ListAgentFirstName",
    "ListAgentLastName",
    "ListAgentFullName",
    "CoListAgentFirstName", 
    "CoListAgentLastName",
    "BuyerAgentAOR",
    "BuyerAgentFirstName", 
    "BuyerAgentLastName",
    "BuyerAgentMlsId",
    "CoBuyerAgentFirstName",
    "ListOfficeName", 
    "BuyerOfficeName", 
    "BuyerOfficeAOR",
    "CoListOfficeName",

    "OriginatingSystemName",
    "OriginatingSystemSubName",

    "StateOrProvince",
    "UnparsedAddress", 
    "StreetNumberNumeric"
]

# Keep UnparsedAddress or StateOrProvince???

# Drop identifier/agent columns from the filtered DataFrame
crmls_sold_filtered_df = crmls_sold_filtered_df.drop(columns=additional_drop_columns)

In [12]:
# Check columns of the filtered DataFrame after dropping the specified columns
print("Columns of the filtered DataFrame after dropping specified columns:")
display(crmls_sold_filtered_df.columns)

Columns of the filtered DataFrame after dropping specified columns:


Index(['Flooring', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN',
       'Latitude', 'Longitude', 'PropertyType', 'LivingArea',
       'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea',
       'MLSAreaMajor', 'TaxAnnualAmount', 'CountyOrParish', 'ElementarySchool',
       'AttachedGarageYN', 'ParkingTotal', 'BuilderName', 'PropertySubType',
       'LotSizeAcres', 'SubdivisionName', 'YearBuilt', 'BathroomsTotalInteger',
       'City', 'TaxYear', 'BuildingAreaTotal', 'BedroomsTotal',
       'ElementarySchoolDistrict', 'ListingContractDate',
       'BelowGradeFinishedArea', 'BusinessType', 'CoveredSpaces',
       'MiddleOrJuniorSchool', 'FireplaceYN', 'Stories', 'HighSchool',
       'Levels', 'LotSizeDimensions', 'LotSizeArea', 'MainLevelBedrooms',
       'NewConstructionYN', 'GarageSpaces', 'HighSchoolDistrict', 'PostalCode',
       'AssociationFee', 'LotSizeSquareFeet', 'MiddleOrJuniorSchoolDistrict',
       'CloseMonth'],
      dtype='object')

## 4.0 . Handle missing values

In [13]:
# Create a summary DataFrame to show the count and percentage of missing values for each column in the filtered DataFrame
missing_summary_df = pd.DataFrame({
    'Count': crmls_sold_filtered_df.isna().sum(),
    'Percentage': crmls_sold_filtered_df.isna().mean() * 100
}).sort_values(by='Percentage', ascending=False)

# Display the missing values summary DataFrame for the filtered DataFrame
display(missing_summary_df)

,Count,Percentage
FireplacesTotal,266809,100.000000
CoveredSpaces,266809,100.000000
AboveGradeFinishedArea,266809,100.000000
MiddleOrJuniorSchoolDistrict,266809,100.000000
ElementarySchoolDistrict,266809,100.000000
WaterfrontYN,266654,99.941906
TaxYear,266642,99.937408
BusinessType,266111,99.738390
TaxAnnualAmount,265856,99.642816
BelowGradeFinishedArea,265410,99.475655


### 4.1. Drop columns with excessive missingness and seem unimportant

- NEED TO EVALUATE IF DROPPED COLUMNS ARE OK TO DROP

In [14]:
# Drop columns with more than 50% missing values from the filtered DataFrame
columns_more_than_50_missing = missing_summary_df[missing_summary_df['Percentage'] > 50].index.tolist()
crmls_sold_filtered_df = crmls_sold_filtered_df.drop(columns=columns_more_than_50_missing)

# Display the number of columns dropped and the names of the columns
print(f"Number of columns dropped due to 50%+ missing values: {len(columns_more_than_50_missing)}")
print(f"Columns dropped due to 50%+ missing values:")
print(columns_more_than_50_missing)

# Display the shape of the filtered DataFrame after dropping columns with more than 50% missing values
print("")
print(f"Shape of the filtered DataFrame after dropping columns with more than 50% missing values: {crmls_sold_filtered_df.shape}") 
print(f"Remaining columns in the filtered DataFrame: {crmls_sold_filtered_df.columns.tolist()}")

Number of columns dropped due to 50%+ missing values: 19
Columns dropped due to 50%+ missing values:
['FireplacesTotal', 'CoveredSpaces', 'AboveGradeFinishedArea', 'MiddleOrJuniorSchoolDistrict', 'ElementarySchoolDistrict', 'WaterfrontYN', 'TaxYear', 'BusinessType', 'TaxAnnualAmount', 'BelowGradeFinishedArea', 'BasementYN', 'BuilderName', 'LotSizeDimensions', 'ElementarySchool', 'MiddleOrJuniorSchool', 'BuildingAreaTotal', 'HighSchool', 'AssociationFeeFrequency', 'SubdivisionName']

Shape of the filtered DataFrame after dropping columns with more than 50% missing values: (266809, 30)
Remaining columns in the filtered DataFrame: ['Flooring', 'ViewYN', 'PoolPrivateYN', 'Latitude', 'Longitude', 'PropertyType', 'LivingArea', 'MLSAreaMajor', 'CountyOrParish', 'AttachedGarageYN', 'ParkingTotal', 'PropertySubType', 'LotSizeAcres', 'YearBuilt', 'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'ListingContractDate', 'FireplaceYN', 'Stories', 'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewCo

In [15]:
# Drop additional columns with many missing values and deemed not useful for analysis from the filtered DataFrame
crmls_sold_filtered_df = crmls_sold_filtered_df.drop(columns=["MainLevelBedrooms", "Flooring", "Levels"])

# Any others that should be dropped? 

### 4.2. Create missing-indicator flag features

- NEED TO EVALUATE IF ANY OTHER COLUMNS SHOULD GET MISSING FLAG!!!

In [16]:
missing_indicator_columns = [
    "LotSizeAcres",
    "LotSizeArea",
    "LotSizeSquareFeet",
    "LivingArea",
    "YearBuilt",
    "GarageSpaces",
    "Stories"
]

# Any others? Keep YearBuilt and LivingArea? Boolean columns? Categorical columns - still do if using encoding?

for col in missing_indicator_columns:
    crmls_sold_filtered_df[f"{col}_missing"] = crmls_sold_filtered_df[col].isna().astype(int)

### 4.3. Impute boolean columns with group-wise mode

- CHANGE GROUP-WISE COLUMN?
- If NewConstructionYN is missing, but YearBuilt matches the year the house was listed or sold, impute it as 1 (True)???
- MAYBE CHANGE THIS TO JUST FALSE IF MISSING? IF SO RETAIN WATERFRONT/BASEMENT?

In [17]:
# Identify boolean columns in the filtered DataFrame and switch their data type to boolean
boolean_columns = crmls_sold_filtered_df.columns[crmls_sold_filtered_df.columns.str.endswith(('YN'))]

# Switch the data type of the identified boolean columns to float
crmls_sold_filtered_df[boolean_columns] = crmls_sold_filtered_df[boolean_columns].astype(float)

# Display the identified boolean columns and their data types after switching to float
print(f"Boolean columns: {list(boolean_columns)}")
display(crmls_sold_filtered_df[boolean_columns].head())
display(crmls_sold_filtered_df[boolean_columns].dtypes)

Boolean columns: ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']


,ViewYN,PoolPrivateYN,AttachedGarageYN,FireplaceYN,NewConstructionYN
0,NaN,NaN,NaN,NaN,NaN
1,1.0,NaN,NaN,NaN,0.0
2,1.0,0.0,NaN,0.0,NaN
3,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,1.0,0.0,0.0


ViewYN               float64
PoolPrivateYN        float64
AttachedGarageYN     float64
FireplaceYN          float64
NewConstructionYN    float64
dtype: object

In [18]:
# Compute the group-wise mode for each boolean column based on the 'PropertySubType' column
# Columns are either 1.0, 0.0 or NaN, so the mode will be either 1.0 or 0.0 for each group
group_modes_boolean = crmls_sold_filtered_df.groupby('PropertySubType')[boolean_columns].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)

# Apply the group-wise mode imputation to the boolean columns
for col in boolean_columns:
    crmls_sold_filtered_df[col] = crmls_sold_filtered_df[col].fillna(crmls_sold_filtered_df['PropertySubType'].map(group_modes_boolean[col]))

    if crmls_sold_filtered_df[col].isna().sum() > 0:
        # If there are still missing values after group-wise mode imputation, fill them with the overall mode
        overall_mode = crmls_sold_filtered_df[col].mode().iloc[0]
        crmls_sold_filtered_df[col] = crmls_sold_filtered_df[col].fillna(overall_mode)

# Check if there are any remaining missing values in the boolean columns after imputation
remaining_missing_values = crmls_sold_filtered_df[boolean_columns].isna().sum()
if remaining_missing_values.sum() == 0:
    print("All missing values in the boolean columns have been successfully imputed.")
else:
    print("There are still some missing values in the boolean columns after imputation.")

All missing values in the boolean columns have been successfully imputed.


### 4.4. Impute numerical columns with group-wise mode/median

- EVALUATE IMPUTATION CHOICE OF MODE vs. MEDIAN 
- NEED TO FINISH IMPUTING FOR YEAR AND ASSOCIATION FEE!!!

In [19]:
# Identify numeric columns that are not boolean columns
numeric_columns = crmls_sold_filtered_df.select_dtypes(include=['number']).columns.difference(boolean_columns)
print(f"Numeric columns in the filtered DataFrame: {list(numeric_columns)}")                                               

Numeric columns in the filtered DataFrame: ['AssociationFee', 'BathroomsTotalInteger', 'BedroomsTotal', 'GarageSpaces', 'GarageSpaces_missing', 'Latitude', 'LivingArea', 'LivingArea_missing', 'Longitude', 'LotSizeAcres', 'LotSizeAcres_missing', 'LotSizeArea', 'LotSizeArea_missing', 'LotSizeSquareFeet', 'LotSizeSquareFeet_missing', 'ParkingTotal', 'Stories', 'Stories_missing', 'YearBuilt', 'YearBuilt_missing']


In [20]:
# # Impute missing values in the "AssociationFee" column
# crmls_sold_filtered_df["AssociationFee"] = crmls_sold_filtered_df["AssociationFee"].fillna(0)

In [21]:
# # Impute year based on group-wise median/mode for the "YearBuilt" column based on the "PostalCode" column???

In [22]:
# Identify numeric columns to impute using group-wise mode based on the 'PropertySubType' column
mode_impute_columns = ["Stories", "BathroomsTotalInteger", "BedroomsTotal", "ParkingTotal", "GarageSpaces"]

# Compute the group-wise mode for each numeric column to be imputed based on the 'PropertySubType' column
group_modes_numeric = crmls_sold_filtered_df.groupby('PropertySubType')[mode_impute_columns].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)

# Apply the group-wise mode imputation to the numeric columns
for col in mode_impute_columns:
    crmls_sold_filtered_df[col] = crmls_sold_filtered_df[col].fillna(crmls_sold_filtered_df['PropertySubType'].map(group_modes_numeric[col]))

    if crmls_sold_filtered_df[col].isna().sum() > 0:
        # If there are still missing values after group-wise mode imputation, fill them with the overall mode
        overall_mode = crmls_sold_filtered_df[col].mode().iloc[0]
        crmls_sold_filtered_df[col] = crmls_sold_filtered_df[col].fillna(overall_mode)

# Check if there are any remaining missing values in the numeric columns after imputation
remaining_missing_values = crmls_sold_filtered_df[mode_impute_columns].isna().sum()
if remaining_missing_values.sum() == 0:
    print("All missing values in the numeric columns have been successfully imputed.")
else:
    print("There are still missing values in the numeric columns after imputation.")

All missing values in the numeric columns have been successfully imputed.


In [23]:
# Identify numeric columns to impute using group-wise median based on the 'PropertySubType' column
median_impute_columns = ["LotSizeAcres", "LotSizeSquareFeet", "LotSizeArea", "LivingArea"]

# Compute the group-wise median for each numeric column to be imputed based on the 'PropertySubType' column
group_medians_numeric = crmls_sold_filtered_df.groupby('PropertySubType')[median_impute_columns].median()

# Apply the group-wise median imputation to the numeric columns
for col in median_impute_columns:
    crmls_sold_filtered_df[col] = crmls_sold_filtered_df[col].fillna(crmls_sold_filtered_df['PropertySubType'].map(group_medians_numeric[col]))

    if crmls_sold_filtered_df[col].isna().sum() > 0:
        # If there are still missing values after group-wise median imputation, fill them with the overall median
        overall_median = crmls_sold_filtered_df[col].median()
        crmls_sold_filtered_df[col] = crmls_sold_filtered_df[col].fillna(overall_median)

# Check if there are any remaining missing values in the numeric columns after imputation
remaining_missing_values = crmls_sold_filtered_df[median_impute_columns].isna().sum()
if remaining_missing_values.sum() == 0:
    print("All missing values in the numeric columns have been successfully imputed.")
else:
    print("There are still missing values in the numeric columns after imputation.")

All missing values in the numeric columns have been successfully imputed.


### 4.5. Impute categorical variables with "missing"

In [24]:
# Identify categorical columns that are not boolean columns
categorical_columns = crmls_sold_filtered_df.select_dtypes(include=['object']).columns
print(f"Categorical columns in the filtered DataFrame: {list(categorical_columns)}")

Categorical columns in the filtered DataFrame: ['PropertyType', 'MLSAreaMajor', 'CountyOrParish', 'PropertySubType', 'City', 'HighSchoolDistrict', 'PostalCode']


In [25]:
# Impute missing values in the "HighSchoolDistrict", "MLSAreaMajor", and "PropertySubType" columns with "missing"
crmls_sold_filtered_df["HighSchoolDistrict"] = crmls_sold_filtered_df["HighSchoolDistrict"].fillna("missing")
crmls_sold_filtered_df["MLSAreaMajor"] = crmls_sold_filtered_df["MLSAreaMajor"].fillna("missing")
crmls_sold_filtered_df["PropertySubType"] = crmls_sold_filtered_df["PropertySubType"].fillna("missing")

# Do not believe these will need to have missing flag - redundancy when encoding categorical columns

### 4.5. Drop subset of data for columns with limited null values

Some of these are columns that could not be reasonably imputed as well

In [26]:
# Print the number of rows to be dropped due to missing values in the specified columns
rows_dropped = len(crmls_sold_filtered_df) - len(crmls_sold_filtered_df.dropna(subset=["PostalCode", "Latitude", "Longitude", "City", "ListingContractDate"]))
print(f"Number of rows dropped due to missing values in PostalCode, Latitude, Longitude, City, or ListingContractDate: {rows_dropped}")

# Drop subset of data for rows in which any of the following columns are null: PostalCode, Latitude, Longitude, City, and ListingContractDate
crmls_sold_filtered_df = crmls_sold_filtered_df.dropna(subset=["PostalCode", "Latitude", "Longitude", "City", "ListingContractDate"])

Number of rows dropped due to missing values in PostalCode, Latitude, Longitude, City, or ListingContractDate: 2210


In [27]:
# Create a new missing value summary DataFrame
missing_summary_df = pd.DataFrame({
    'Count': crmls_sold_filtered_df.isna().sum(),
    'Percentage': crmls_sold_filtered_df.isna().mean() * 100
}).sort_values(by='Percentage', ascending=False)    

# Display the missing values summary DataFrame for the filtered DataFrame after all imputations and row drops
display(missing_summary_df)

,Count,Percentage
AssociationFee,80702,30.499737
YearBuilt,10313,3.897596
ViewYN,0,0.000000
LotSizeSquareFeet,0,0.000000
NewConstructionYN,0,0.000000
GarageSpaces,0,0.000000
HighSchoolDistrict,0,0.000000
PostalCode,0,0.000000
CloseMonth,0,0.000000
Stories,0,0.000000


## 5.0. Train/test splitting

In [28]:
def create_train_test_split(df, X):
    """
    Create a train-test split based on the 'CloseMonth' column in the DataFrame.
    The most recent month of available data is used as the test set, and the X
    months immediately preceding it are used as the training set. X is a tunable parameter.
    """
    train_end_month = df['CloseMonth'].max()
    train_start_month = train_end_month - X  # subtract integer months from a Period
    train_df = df[(df['CloseMonth'] >= train_start_month) & (df['CloseMonth'] < train_end_month)]
    test_df = df[df['CloseMonth'] == train_end_month]

    return train_df.drop(columns=['CloseMonth']), test_df.drop(columns=['CloseMonth'])

# Identify possible X values for the train-test split
possible_X_values = list(range(1, len(crmls_sold_filtered_df['CloseMonth'].unique())))

# Test the train-test split with different X values
for X in possible_X_values:
    train_df, test_df = create_train_test_split(crmls_sold_filtered_df, X)
    print(f" -- TRAIN/TEST SETS for X = {X} Months: -- ")
    print(f"Train set shape: {train_df.shape}, Test set shape: {test_df.shape}")

 -- TRAIN/TEST SETS for X = 1 Months: -- 
Train set shape: (22847, 33), Test set shape: (21857, 33)
 -- TRAIN/TEST SETS for X = 2 Months: -- 
Train set shape: (48008, 33), Test set shape: (21857, 33)
 -- TRAIN/TEST SETS for X = 3 Months: -- 
Train set shape: (73542, 33), Test set shape: (21857, 33)
 -- TRAIN/TEST SETS for X = 4 Months: -- 
Train set shape: (97561, 33), Test set shape: (21857, 33)
 -- TRAIN/TEST SETS for X = 5 Months: -- 
Train set shape: (121537, 33), Test set shape: (21857, 33)
 -- TRAIN/TEST SETS for X = 6 Months: -- 
Train set shape: (144652, 33), Test set shape: (21857, 33)
 -- TRAIN/TEST SETS for X = 7 Months: -- 
Train set shape: (163442, 33), Test set shape: (21857, 33)
 -- TRAIN/TEST SETS for X = 8 Months: -- 
Train set shape: (179923, 33), Test set shape: (21857, 33)
 -- TRAIN/TEST SETS for X = 9 Months: -- 
Train set shape: (200447, 33), Test set shape: (21857, 33)
 -- TRAIN/TEST SETS for X = 10 Months: -- 
Train set shape: (219524, 33), Test set shape: (2185

Choose X = 4

In [29]:
X = 4
train_df, test_df = create_train_test_split(crmls_sold_filtered_df, X)
print(f"Train set shape: {train_df.shape}, Test set shape: {test_df.shape}")

Train set shape: (97561, 33), Test set shape: (21857, 33)


##  6.0. Categorical encoding

I will use target encoding computed with cross-validation folds for these variables after conducting test/train split

- EVALUATE WHICH COLUMNS TO USE IN TARGET ENCODING? FOR NON USED, USE ONE HOT ENCODING INSTEAD
- HANDLE LISTCONTRACTDATE COLUMN

In [ ]:
# Keep these or exclude 'PropertyType'/'PropertySubType' - do OneHot instead???

high_cardinality_categorical_columns = [
    "PropertyType"
    "PropertySubType",
    "City",
    "PostalCode",
    "CountyOrParish",
    "MLSAreaMajor",
    "HighSchoolDistrict"
]

# Initialize the TargetEncoder with smoothing and a random state for reproducibility
target_encoder = TargetEncoder(smooth="auto", random_state=42)

# Fit the TargetEncoder on the training set and transform both the training and test sets for the high-cardinality categorical columns
train_df[high_cardinality_categorical_columns] = target_encoder.fit_transform(train_df[high_cardinality_categorical_columns], train_df['ClosePrice'])
test_df[high_cardinality_categorical_columns] = target_encoder.transform(test_df[high_cardinality_categorical_columns])

In [ ]:
# Save newly preprocessed DataFrame to a CSV file for future use
# preprocessed_csv_filepath = os.path.join(assets_directory, "crmls_sold_preprocessed.csv")
# crmls_sold_filtered_df.to_csv(preprocessed_csv_filepath, index=False)